# Chapter 14 — Riemannian Metrics and Optimization on Manifolds

Companion notebook to Chapter 14 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

Two replacements run through the chapter: the gradient `G^{-1} [df]^T`
instead of `[df]^T`, and a retraction `R_p(v)` instead of `p + v`.

In [1]:
import math
import warnings
import torch
from torch.func import jacrev, jacfwd, hessian

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
print(torch.__version__)
T = torch.tensor
unit = lambda q: q / q.norm()
psiN = lambda t: torch.cat([2 * t, (t @ t - 1).reshape(1)]) / (t @ t + 1)   # stereographic, from the north pole
psiS = lambda t: torch.cat([2 * t, (1 - t @ t).reshape(1)]) / (t @ t + 1)
phiN = lambda q: q[:-1] / (1 - q[-1])
phiS = lambda q: q[:-1] / (1 + q[-1])
tauNS = lambda t: t / (t @ t)
def gram(psi, t):
    J = jacfwd(psi)(t)
    return J.T @ J

2.15.0+cu132


## 1. Riemannian metrics

Example 14.2: Gram matrices in a chart, the rule `G' = J^{-T} G J^{-1}`, and
the induced metric of the sphere in a stereographic chart (Exercise 14.2).

In [2]:
cart = lambda rt: torch.stack([rt[0] * torch.cos(rt[1]), rt[0] * torch.sin(rt[1])])
rt = T([2.0, 0.7])
Gpol = gram(cart, rt)
print("polar chart, r = 2:  G =", [[round(v, 6) + 0.0 for v in row] for row in Gpol.tolist()])
assert torch.allclose(Gpol, torch.diag(T([1.0, 4.0])))

for dim in (2, 3):                                              # S^2 and S^3
    t = torch.randn(dim)
    assert torch.allclose(gram(psiN, t), 4 / (1 + t @ t) ** 2 * torch.eye(dim))
t = T([0.7, -1.2]); tS = tauNS(t)
J = jacrev(tauNS)(t); Ji = torch.linalg.inv(J)
assert torch.allclose(psiN(t), psiS(tS))                        # the same point of the sphere
assert torch.allclose(gram(psiS, tS), Ji.T @ gram(psiN, t) @ Ji)
print("sphere: G(t) = 4 / (1 + |t|^2)^2 I, and G' = J^-T G J^-1 between the two stereographic charts")

# the metric of a chart: identity in that chart, J^-T J^-1 in another
Gchart = Ji.T @ torch.eye(2) @ Ji
print("metric of the north chart seen from the south chart:", [[round(v, 4) + 0.0 for v in row] for row in Gchart.tolist()])
assert not torch.allclose(Gchart, torch.eye(2))

polar chart, r = 2:  G = [[1.0, 0.0], [0.0, 4.0]]


sphere: G(t) = 4 / (1 + |t|^2)^2 I, and G' = J^-T G J^-1 between the two stereographic charts
metric of the north chart seen from the south chart: [[3.7249, 0.0], [0.0, 3.7249]]


## 2. Lengths in the hyperbolic disc

Example 14.4 and Figure 14.1: the distance from the centre is
`2 artanh |x|`; other curves between the same points are longer.

In [3]:
lam = lambda x: 2 / (1 - x @ x)
def length(curve, n=20001):
    ts = torch.linspace(0.0, 1.0, n)
    pts = torch.stack([curve(t) for t in ts])
    mids = 0.5 * (pts[1:] + pts[:-1])
    seg = (pts[1:] - pts[:-1]).norm(dim=1)
    return (2 / (1 - (mids ** 2).sum(1)) * seg).sum().item()
x = T([0.6, 0.3])
straight = lambda t: t * x
wiggle = lambda t: t * x + 0.15 * torch.sin(math.pi * t) * T([-0.3, 0.6])
bent = lambda t: t * x + 0.3 * t * (1 - t) * T([1.0, -1.0])
d0 = 2 * math.atanh(x.norm().item())
print("2 artanh|x| =", round(d0, 6), "  straight segment:", round(length(straight), 6),
      "  two other curves:", round(length(wiggle), 4), round(length(bent), 4))
assert abs(length(straight) - d0) < 1e-6 and length(wiggle) > d0 and length(bent) > d0
for rho in (0.9, 0.99, 0.999):
    print(f"Euclidean distance {rho} from the centre  ->  hyperbolic distance {2 * math.atanh(rho):.3f}")

2 artanh|x| = 1.624467   straight segment: 1.624467   two other curves: 1.7245 1.7397


Euclidean distance 0.9 from the centre  ->  hyperbolic distance 2.944
Euclidean distance 0.99 from the centre  ->  hyperbolic distance 5.293
Euclidean distance 0.999 from the centre  ->  hyperbolic distance 7.600


## 3. The Riemannian gradient

Proposition 14.6: `[grad f] = G^{-1} [df]^T` in a chart, it changes with the
chart as a vector, it is the projection of the ambient gradient for an
induced metric, and it is the direction of steepest ascent. Then the polar
example, Example 14.8 (the Rayleigh quotient) and Exercises 14.1 and 14.2.

In [4]:
A3 = T([[2.0, 0.5, 0.0], [0.5, 1.0, 0.3], [0.0, 0.3, -1.0]])
Fext = lambda q: q @ A3 @ q + torch.sin(q[0])                   # a function on R^3, restricted to the sphere
t = T([0.7, -1.2]); tS = tauNS(t); x = psiN(t)
rowN, rowS = jacrev(lambda u: Fext(psiN(u)))(t), jacrev(lambda u: Fext(psiS(u)))(tS)
gN = torch.linalg.solve(gram(psiN, t), rowN)                    # G^{-1} [df]^T in the north chart
gS = torch.linalg.solve(gram(psiS, tS), rowS)
J = jacrev(tauNS)(t)
assert torch.allclose(gS, J @ gN)                               # a tangent vector
assert not torch.allclose(rowS, J @ rowN)                       # the transposed row is not
Pi = torch.eye(3) - torch.outer(x, x)
amb = Pi @ jacrev(Fext)(x)                                      # projection of the ambient gradient
assert torch.allclose(jacfwd(psiN)(t) @ gN, amb) and torch.allclose(jacfwd(psiS)(tS) @ gS, amb)
print("grad f in R^3:", [round(v, 4) for v in amb.tolist()], " from both charts and from the projection")
# steepest ascent among unit tangent vectors
best = 0.0
for _ in range(2000):
    v = Pi @ torch.randn(3); v = v / v.norm()
    best = max(best, (jacrev(Fext)(x) @ v).item())
print("largest df(v) over 2000 unit tangent vectors:", round(best, 4), " |grad f| =", round(amb.norm().item(), 4))
assert best <= amb.norm().item() + 1e-12 and best > 0.99 * amb.norm().item()

# polar coordinates: the missing G^{-1}
rt = T([2.0, math.pi / 6])
row = jacrev(lambda u: cart(u)[0])(rt)
col = torch.linalg.solve(gram(cart, rt), row)
assert torch.allclose(col, torch.linalg.solve(jacrev(cart)(rt), T([1.0, 0.0])))
print("polar chart: G^{-1}[dx]^T =", [round(v, 4) for v in col.tolist()], "= components of d/dx")

# Rayleigh quotient: the gradient vanishes exactly at the eigenvectors
ray = lambda q: q @ A3 @ q
gradR = lambda q: 2 * (A3 @ q - (q @ A3 @ q) * q)
lam3, U3 = torch.linalg.eigh(A3)
assert max(gradR(U3[:, i]).norm().item() for i in range(3)) < 1e-12
q = unit(T([1.0, 2.0, 2.0]))
assert torch.allclose(gradR(q), (torch.eye(3) - torch.outer(q, q)) @ jacrev(ray)(q)) and gradR(q).norm() > 0.1

# Exercise 14.1
f1 = lambda u: u[0] ** 2 * u[1]
rt = T([1.5, 0.4]); row = jacrev(f1)(rt); g1 = torch.linalg.solve(torch.diag(T([1.0, rt[0] ** 2])), row)
assert torch.allclose(row, T([2 * 1.5 * 0.4, 1.5 ** 2])) and torch.allclose(g1, T([2 * 1.5 * 0.4, 1.0]))
assert abs((row @ g1).item() - (4 * 1.5 ** 2 * 0.4 ** 2 + 1.5 ** 2)) < 1e-12
# Exercise 14.2: the gradient of the height has the components t
t = T([0.7, -1.2]); rowh = jacrev(lambda u: psiN(u)[-1])(t)
assert torch.allclose(rowh, 4 * t / (1 + t @ t) ** 2)
assert torch.allclose(torch.linalg.solve(gram(psiN, t), rowh), t)
print("Exercises 14.1 and 14.2 checked; in the stereographic chart [grad height] = t")

grad f in R^3: [1.3193, 0.163, -1.5653]  from both charts and from the projection


largest df(v) over 2000 unit tangent vectors: 2.0536  |grad f| = 2.0536
polar chart: G^{-1}[dx]^T = [0.866, -0.25] = components of d/dx
Exercises 14.1 and 14.2 checked; in the stereographic chart [grad height] = t


## 4. Retractions

Definition 14.9, Proposition 14.10 and Example 14.11, with Figure 14.2 and
Exercises 14.3 and 14.6.

In [5]:
s0 = T(0.0)
R_sph = lambda x, v: (x + v) / (x + v).norm()
def Exp_sph(x, v):
    # cos|v| x + (sin|v| / |v|) v, written so that it can be differentiated at v = 0
    m = v @ v
    big = m > 1e-8
    n = torch.sqrt(torch.where(big, m, torch.ones_like(m)))
    cosn = torch.where(big, torch.cos(n), 1 - m / 2 + m * m / 24)
    sinc = torch.where(big, torch.sin(n) / n, 1 - m / 6 + m * m / 120)
    return cosn * x + sinc * v
x = unit(T([1.0, 2.0, 2.0])); v = (torch.eye(3) - torch.outer(x, x)) @ T([0.3, -1.0, 0.4])
for R in (R_sph, Exp_sph):
    c = lambda s: R(x, s * v)
    assert torch.allclose(c(s0), x) and torch.allclose(jacfwd(c)(s0), v)
    assert abs(R(x, v).norm().item() - 1) < 1e-12
# Figure 14.2
p2, v2 = T([1.0, 0.0]), T([0.0, 1.2])
print("Figure 14.2: R_p(v) =", [round(a, 3) for a in R_sph(p2, v2).tolist()], "  Exp_p(v) =", [round(a, 3) for a in Exp_sph(p2, v2).tolist()])
assert torch.allclose(R_sph(p2, v2), T([0.640, 0.768]), atol=5e-4) and torch.allclose(Exp_sph(p2, v2), T([0.362, 0.932]), atol=5e-4)
# Exercise 14.3: R - Exp = -(1/3)|v|^2 v + O(|v|^4)
errs = []
for e in (0.2, 0.1, 0.05):
    w = e * v
    errs.append((R_sph(x, w) - Exp_sph(x, w) + (w @ w) * w / 3).norm().item())
print("|R - Exp + |v|^2 v / 3| for |v| scaled by 0.2, 0.1, 0.05:", [f"{a:.1e}" for a in errs])
assert 12 < errs[0] / errs[1] < 20 and 12 < errs[1] / errs[2] < 20           # fourth order

# Stiefel: polar and QR retractions
n, k = 5, 2
Qf, _ = torch.linalg.qr(torch.randn(n, n)); X = Qf[:, :k]
sym = lambda S: 0.5 * (S + S.T)
proj = lambda X, Z: Z - X @ sym(X.T @ Z)
Z = proj(X, torch.randn(n, k))
def polar(X, Z):
    ev, P = torch.linalg.eigh(torch.eye(k) + Z.T @ Z)
    return (X + Z) @ P @ torch.diag(ev ** -0.5) @ P.T
def qf(Y):
    Q, Rm = torch.linalg.qr(Y)
    return Q * torch.sign(torch.diagonal(Rm))
for R in (polar, lambda X, Z: qf(X + Z)):
    Y = R(X, Z)
    assert torch.allclose(Y.T @ Y, torch.eye(k)) and torch.allclose(R(X, 0 * Z), X)
    hstep = 1e-5                                                 # velocity at 0, by central differences
    assert torch.allclose((R(X, hstep * Z) - R(X, -hstep * Z)) / (2 * hstep), Z, atol=1e-8)
assert torch.allclose((X + Z).T @ (X + Z), torch.eye(k) + Z.T @ Z)
print("Stiefel: polar and QR retractions have orthonormal columns and velocity Z at 0")
# orthogonal group: Cayley
Q4, _ = torch.linalg.qr(torch.randn(4, 4)); Om = torch.randn(4, 4); Om = Om - Om.T
cay = lambda s: Q4 @ torch.linalg.inv(torch.eye(4) - 0.5 * s * Om) @ (torch.eye(4) + 0.5 * s * Om)
assert torch.allclose(cay(T(0.8)).T @ cay(T(0.8)), torch.eye(4)) and torch.allclose(jacfwd(cay)(s0), Q4 @ Om)
# Exercise 14.6
X32 = T([[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]]); Z32 = T([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
k = 2
assert torch.allclose(proj(X32, Z32), T([[0.0, -0.5], [0.5, 0.0], [5.0, 6.0]]))
W32 = T([[0.0, 0.0], [0.0, 0.0], [1.0, 0.0]])
assert torch.allclose(polar(X32, W32), T([[1 / math.sqrt(2), 0.0], [0.0, 1.0], [1 / math.sqrt(2), 0.0]]))
print("Exercise 14.6 checked")

Figure 14.2: R_p(v) = [0.64, 0.768]   Exp_p(v) = [0.362, 0.932]
|R - Exp + |v|^2 v / 3| for |v| scaled by 0.2, 0.1, 0.05: ['7.1e-04', '4.5e-05', '2.8e-06']
Stiefel: polar and QR retractions have orthonormal columns and velocity Z at 0
Exercise 14.6 checked


## 5. Riemannian gradient descent

Theorem 14.12 and Proposition 14.13 on the sphere, Example 14.14 (a power
iteration with a shift), Exercise 14.5, then Proposition 14.15 and
Example 14.16 on the Stiefel manifold.

In [6]:
torch.manual_seed(1)
n = 6
B = torch.randn(n, n); A = 0.5 * (B + B.T)
lamA, UA = torch.linalg.eigh(A)                                  # increasing eigenvalues
f = lambda q: q @ A @ q
grad = lambda q: 2 * (A @ q - (q @ A @ q) * q)
# a constant L for the descent inequality: second derivative of t -> f(R_x(t u)), sampled
h2 = jacfwd(jacfwd(lambda t, q, u: f(R_sph(q, t * u))))
Lnum = 0.0
for _ in range(400):
    q = unit(torch.randn(n)); u = torch.randn(n); u = unit(u - (u @ q) * q)
    for tv in (0.0, 0.3, 0.7, 1.5, 3.0):
        Lnum = max(Lnum, abs(h2(T(tv), q, u).item()))
L = 1.2 * Lnum
print("sampled bound on the second derivative along the retraction:", round(Lnum, 3), "  L =", round(L, 3))
q = unit(torch.randn(n)); f0 = f(q).item(); fstar = lamA[0].item(); hist = []
for kk in range(400):
    g = grad(q); s = -g / L
    q_new = R_sph(q, s)
    assert f(q_new) <= f(q) + (2 * (A @ q)) @ s + 0.5 * L * (s @ s) + 1e-12       # the descent inequality
    assert f(q_new) <= f(q) - (g @ g) / (2 * L) + 1e-12                           # the decrease of the theorem
    hist.append((g @ g).item()); q = q_new
for K in (10, 100, 400):
    assert min(hist[:K]) <= 2 * L * (f0 - fstar) / K
print("f after 400 steps:", round(f(q).item(), 8), "  smallest eigenvalue:", round(fstar, 8),
      "  |<x, u_min>| =", round(abs((q @ UA[:, 0]).item()), 8))
assert abs(f(q).item() - fstar) < 1e-6

# Example 14.14: one ascent step = one power step for A + c I
q = unit(torch.randn(n)); eta = 0.3; rho = f(q).item()
asc = R_sph(q, eta * grad(q))
c = (1 - 2 * eta * rho) / (2 * eta)
pw = (A + c * torch.eye(n)) @ q
assert torch.allclose(asc, unit(pw) * torch.sign(pw @ asc))
# Exercise 14.5
A2 = torch.diag(T([3.0, 1.0])); x0 = T([1.0, 1.0]) / math.sqrt(2)
g0 = 2 * (A2 @ x0 - (x0 @ A2 @ x0) * x0)
x1 = R_sph(x0, 0.25 * g0)
print("Exercise 14.5: grad =", [round(v, 4) for v in g0.tolist()], "  x1 =", [round(v, 4) for v in x1.tolist()],
      "  f:", round((x0 @ A2 @ x0).item(), 6), "->", round((x1 @ A2 @ x1).item(), 6))
assert torch.allclose(g0, math.sqrt(2) * T([1.0, -1.0])) and torch.allclose(x1, T([3.0, 1.0]) / math.sqrt(10))
assert abs((x1 @ A2 @ x1).item() - 2.8) < 1e-12 and torch.allclose(x1, unit(A2 @ x0))

# Stiefel: the projection, and principal components by Riemannian gradient ascent
k = 2
Qf, _ = torch.linalg.qr(torch.randn(n, n)); X = Qf[:, :k]
Z1, Z2 = torch.randn(n, k), torch.randn(n, k)
P1 = proj(X, Z1)
assert torch.allclose(proj(X, P1), P1)                                           # idempotent
assert torch.allclose(X.T @ P1 + P1.T @ X, torch.zeros(k, k), atol=1e-12)        # tangent
assert abs(((Z1 - P1) * proj(X, Z2)).sum().item()) < 1e-12                       # Z - Pi Z is normal
assert abs((proj(X, Z1) * Z2).sum().item() - (Z1 * proj(X, Z2)).sum().item()) < 1e-12   # self-adjoint
C = B @ B.T / n                                                                   # a covariance matrix
lamC, UC = torch.linalg.eigh(C)
fS = lambda Y: torch.trace(Y.T @ C @ Y)
gS = lambda Y: 2 * (C @ Y - Y @ (Y.T @ C @ Y))
assert torch.allclose(gS(X), proj(X, jacrev(fS)(X)))
Qk, _ = torch.linalg.qr(torch.randn(k, k))
assert abs((fS(X @ Qk) - fS(X)).item()) < 1e-12                                   # depends on the subspace only
Y = X.clone(); step = 0.25 / lamC[-1].item()
for _ in range(3000):
    Y = qf(Y + step * gS(Y))
top = UC[:, -k:]
print("captured variance:", round(fS(Y).item(), 8), "  sum of the two largest eigenvalues:", round(lamC[-k:].sum().item(), 8))
assert abs(fS(Y).item() - lamC[-k:].sum().item()) < 1e-8
assert torch.allclose(Y @ Y.T, top @ top.T, atol=1e-4)                            # the same subspace
assert torch.allclose(C @ Y, Y @ (Y.T @ C @ Y), atol=1e-4)                        # stable by C

sampled bound on the second derivative along the retraction: 5.119   L = 6.142
f after 400 steps: -1.10947805   smallest eigenvalue: -1.10947805   |<x, u_min>| = 1.0
Exercise 14.5: grad = [1.4142, -1.4142]   x1 = [0.9487, 0.3162]   f: 2.0 -> 2.8
captured variance: 2.62006218   sum of the two largest eigenvalues: 2.62006218


## 6. The Hessian at a critical point

Proposition 14.17 and Theorem 14.18, with Exercise 14.9: second derivatives
along any retraction, the eigenvalues `2 (lambda_j - lambda_i)`, and their
independence of the chart.

In [7]:
A3d = torch.diag(T([3.0, 2.0, 1.0]))
fr = lambda q: q @ A3d @ q
E = torch.eye(3)
for i, (lam_i, nature) in enumerate(((3.0, "maximum"), (2.0, "saddle"), (1.0, "minimum"))):
    u_i = E[:, i]
    others = [j for j in range(3) if j != i]
    Bt = E[:, others]                                             # orthonormal basis of the tangent space
    v = unit(Bt @ T([0.6, -0.8]))
    for R in (R_sph, Exp_sph):
        d2 = jacfwd(jacfwd(lambda s: fr(R(u_i, s * v))))(T(0.0)).item()
        assert abs(d2 - 2 * ((v @ A3d @ v).item() - lam_i)) < 1e-10
    ev = torch.linalg.eigvalsh(Bt.T @ (2 * (A3d - lam_i * E)) @ Bt)
    print(f"critical point e{i + 1}: eigenvalues of the Hessian {ev.tolist()}  ({nature})")
    assert torch.allclose(ev, T(sorted(2 * (A3d[j, j].item() - lam_i) for j in others)))
# eigenvalues of G^{-1} H in two charts, at a critical point of another Rayleigh quotient
fa = lambda q: q @ A3 @ q
ui = U3[:, 0]; expected = torch.sort(2 * (lam3[1:] - lam3[0])).values
for psi, phi in ((psiN, phiN), (psiS, phiS)):
    tc = phi(ui)
    assert jacrev(lambda u: fa(psi(u)))(tc).abs().max() < 1e-12            # critical in the chart too
    Hc = hessian(lambda u: fa(psi(u)))(tc)
    evc = torch.sort(torch.linalg.eigvals(torch.linalg.solve(gram(psi, tc), Hc)).real).values
    assert torch.allclose(evc, expected, atol=1e-9)
    print("eigenvalues of G^{-1} H in a chart:", [round(v, 6) for v in evc.tolist()],
          "   of H alone:", [round(v, 4) for v in torch.linalg.eigvalsh(Hc).tolist()])
print("2 (lambda_j - lambda_i):", [round(v, 6) for v in expected.tolist()])

critical point e1: eigenvalues of the Hessian [-4.0, -2.0]  (maximum)
critical point e2: eigenvalues of the Hessian [-2.0, 2.0]  (saddle)
critical point e3: eigenvalues of the Hessian [2.0, 4.0]  (minimum)
eigenvalues of G^{-1} H in a chart: [3.76076, 6.514224]    of H alone: [0.0005, 0.0009]
eigenvalues of G^{-1} H in a chart: [3.76076, 6.514224]    of H alone: [14.8665, 25.751]
2 (lambda_j - lambda_i): [3.76076, 6.514224]


## 7. Geodesics and flatness

Definition 14.19 on the sphere and on the cylinder (Exercise 14.8),
Example 14.22, Proposition 14.23 and the curvature formula (Figure 14.3).

In [8]:
x = unit(T([1.0, 2.0, 2.0])); v = (torch.eye(3) - torch.outer(x, x)) @ T([0.3, -1.0, 0.4])
great = lambda s: Exp_sph(x, s * v)
for sv in (0.2, 1.0, 2.5):
    c = great(T(sv)); acc = jacfwd(jacfwd(great))(T(sv))
    tangential = acc - (acc @ c) * c
    assert tangential.norm() < 1e-9 and torch.allclose(acc, -(v @ v) * c, atol=1e-9)
    assert abs(jacfwd(great)(T(sv)).norm().item() - v.norm().item()) < 1e-10        # constant speed
small = lambda s: torch.stack([0.6 * torch.cos(s), 0.6 * torch.sin(s), torch.tensor(0.8)])   # a parallel, not a great circle
c = small(T(0.4)); acc = jacfwd(jacfwd(small))(T(0.4))
print("great circle: no tangential acceleration;  small circle: tangential acceleration of norm",
      round((acc - (acc @ c) * c).norm().item(), 4))
assert (acc - (acc @ c) * c).norm() > 0.1

# cylinder: flat chart, helices are geodesics
cyl = lambda u: torch.stack([torch.cos(u[0]), torch.sin(u[0]), u[1]])
assert torch.allclose(gram(cyl, T([0.9, -0.4])), torch.eye(2))
a_, b_ = 1.3, 0.7
helix = lambda s: cyl(torch.stack([a_ * s, b_ * s]))
c = helix(T(0.6)); acc = jacfwd(jacfwd(helix))(T(0.6)); nrm = T([c[0].item(), c[1].item(), 0.0])
assert torch.allclose(acc, -a_ ** 2 * nrm)
# the embedded torus of Exercise 12.7
torus = lambda z: torch.stack([(2 + torch.cos(z[1])) * torch.cos(z[0]), (2 + torch.cos(z[1])) * torch.sin(z[0]), torch.sin(z[1])])
zt = T([0.7, 1.1])
assert torch.allclose(gram(torus, zt), torch.diag(T([(2 + math.cos(1.1)) ** 2, 1.0])))
print("cylinder: G = I in the chart (theta, z);  embedded torus: G = diag((2 + cos phi)^2, 1)")

# circles: sphere, flat, hyperbolic, and the curvature formula
def K_estimate(C, r):
    return 3 * (2 * math.pi * r - C) / (math.pi * r ** 3)
for r in (0.4, 0.2, 0.1):
    th = torch.linspace(0, 2 * math.pi, 4001)
    north = T([0.0, 0.0, 1.0])
    circ = torch.stack([math.sin(r) * torch.cos(th), math.sin(r) * torch.sin(th), math.cos(r) * torch.ones_like(th)], 1)
    assert torch.allclose(torch.acos(circ @ north), r * torch.ones_like(th), atol=1e-7)   # at distance r from the pole
    C_sph = (circ[1:] - circ[:-1]).norm(dim=1).sum().item()
    rho = math.tanh(r / 2)
    C_hyp = 2 / (1 - rho ** 2) * 2 * math.pi * rho
    assert abs(C_sph - 2 * math.pi * math.sin(r)) < 1e-5 and abs(C_hyp - 2 * math.pi * math.sinh(r)) < 1e-12
    print(f"r = {r}: sphere {C_sph:.5f}  flat {2 * math.pi * r:.5f}  hyperbolic {C_hyp:.5f}   "
          f"curvature estimates {K_estimate(C_sph, r):+.4f}, {K_estimate(2 * math.pi * r, r):+.4f}, {K_estimate(C_hyp, r):+.4f}")
assert abs(K_estimate(2 * math.pi * math.sin(0.1), 0.1) - 1) < 2e-3 and abs(K_estimate(2 * math.pi * math.sinh(0.1), 0.1) + 1) < 2e-3

great circle: no tangential acceleration;  small circle: tangential acceleration of norm 0.48
cylinder: G = I in the chart (theta, z);  embedded torus: G = diag((2 + cos phi)^2, 1)
r = 0.4: sphere 2.44679  flat 2.51327  hyperbolic 2.58083   curvature estimates +0.9920, +0.0000, -1.0080
r = 0.2: sphere 1.24828  flat 1.25664  hyperbolic 1.26503   curvature estimates +0.9980, +0.0000, -1.0020
r = 0.1: sphere 0.62727  flat 0.62832  hyperbolic 0.62937   curvature estimates +0.9996, +0.0000, -1.0005


## 8. Machine learning

The two boxes of Section 14.7 and Exercises 14.4 and 14.7: weight
normalization as Riemannian descent on the sphere, Oja's rule, the gradient
in the hyperbolic disc, and the metric of a chart on the Gaussian family.

In [9]:
# weight normalization: a gradient step on v, then normalization = a Riemannian step on the sphere
torch.manual_seed(3)
Xd, yd = torch.randn(30, 5), torch.randn(30)
ell = lambda w: ((Xd @ w - yd) ** 2).mean()                       # a loss of the direction w
Lv = lambda v: ell(v / v.norm())
v = 1.7 * unit(torch.randn(5)); w = v / v.norm(); eta = 0.3
v_new = v - eta * jacrev(Lv)(v)
rgrad = (torch.eye(5) - torch.outer(w, w)) @ jacrev(ell)(w)
assert torch.allclose(v_new / v_new.norm(), R_sph(w, -eta / (v @ v) * rgrad))
print("weight normalization: the new direction is R_w(-(eta/|v|^2) grad f(w)); effective step", round(eta / (v @ v).item(), 4))

# Oja's rule: the update is a Riemannian gradient step for A = x x^T, without the retraction
torch.manual_seed(4)
d = 5
Mx = torch.randn(d, d) @ torch.diag(T([3.0, 1.5, 1.0, 0.6, 0.3]))
cov = Mx @ Mx.T
xs = torch.randn(40000, d) @ Mx.T                                  # samples with covariance cov
w = unit(torch.randn(d))
x0 = xs[0]; y0 = w @ x0
assert torch.allclose(y0 * x0 - y0 ** 2 * w, (torch.eye(d) - torch.outer(w, w)) @ (torch.outer(x0, x0) @ w))
for i, xi in enumerate(xs):
    y = w @ xi
    w = w + (0.02 / (1 + i / 500)) / cov.trace() * (y * xi - y ** 2 * w)
top = torch.linalg.eigh(cov)[1][:, -1]
print("Oja's rule after 40000 samples: |w| =", round(w.norm().item(), 4), "  |<w, first principal direction>| =",
      round(abs((unit(w) @ top).item()), 4))
assert abs(w.norm().item() - 1) < 0.05 and abs((unit(w) @ top).item()) > 0.98

# hyperbolic disc: Exercise 14.7
xh = T([0.6, 0.3])
Gh = lam(xh) ** 2 * torch.eye(2)
gh = torch.linalg.solve(Gh, T([1.0, 0.0]))
assert torch.allclose(gh, (1 - xh @ xh) ** 2 / 4 * T([1.0, 0.0]))
assert abs(math.sqrt((gh @ Gh @ gh).item()) - (1 - (xh @ xh).item()) / 2) < 1e-12
print("hyperbolic disc: grad x1 =", [round(a, 4) for a in gh.tolist()], " at", xh.tolist(),
      ";  distance 3 from the centre is Euclidean distance", round(math.tanh(1.5), 4),
      ";  circle of radius 3:", round(2 * math.pi * math.sinh(3), 2), "against", round(6 * math.pi, 2))
assert abs(2 * math.atanh(math.tanh(1.5)) - 3) < 1e-12

# Exercise 14.4: the chart (mu, ln sigma) as a metric and a retraction on the Gaussian family
Lg = lambda ms: torch.log(ms[1]) + (1 + ms[0] ** 2) / (2 * ms[1] ** 2)
ms0 = T([1.0, 2.0]); eta = 1.2
G = torch.diag(T([1.0, 1 / ms0[1].item() ** 2]))
gr = torch.linalg.solve(G, jacrev(Lg)(ms0))
retr = lambda ms, ab: torch.stack([ms[0] + ab[0], ms[1] * torch.exp(ab[1] / ms[1])])
new = retr(ms0, -eta * gr)
print("Exercise 14.4: grad L =", gr.tolist(), "  Riemannian step ->", [round(a, 4) for a in new.tolist()])
assert torch.allclose(gr, T([0.25, 1.0])) and torch.allclose(new, T([0.7, 2 * math.exp(-0.6)]))
# the same point as gradient descent in the chart (mu, s)
Ls = lambda u: Lg(torch.stack([u[0], torch.exp(u[1])]))
u0 = torch.stack([ms0[0], torch.log(ms0[1])]); u1 = u0 - eta * jacrev(Ls)(u0)
assert torch.allclose(new, torch.stack([u1[0], torch.exp(u1[1])]))

weight normalization: the new direction is R_w(-(eta/|v|^2) grad f(w)); effective step 0.1038


Oja's rule after 40000 samples: |w| = 1.0   |<w, first principal direction>| = 1.0
hyperbolic disc: grad x1 = [0.0756, 0.0]  at [0.6, 0.3] ;  distance 3 from the centre is Euclidean distance 0.9051 ;  circle of radius 3: 62.94 against 18.85
Exercise 14.4: grad L = [0.25, 1.0]   Riemannian step -> [0.7, 1.0976]


## Summary

* The Gram matrix of a metric changes as `J^{-T} G J^{-1}`; the column
  `G^{-1} [df]^T` is a tangent vector, the transposed row is not.
* For an induced metric the gradient is the projection of the ambient
  gradient.
* Riemannian gradient descent on the sphere satisfies the same decrease and
  the same rate as gradient descent, with a constant `L` for the function
  seen through the retraction; it finds the smallest eigenvalue, and on the
  Stiefel manifold the principal subspace.
* At a critical point the eigenvalues of `G^{-1} H` do not depend on the
  chart; for the Rayleigh quotient they are `2 (lambda_j - lambda_i)`.
* Circles of radius `r` have length `2 pi sin r`, `2 pi r` and
  `2 pi sinh r` on the sphere, in a flat manifold and in the hyperbolic disc.